In [17]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
from pathlib import Path

import pandas as pd

In [15]:
file_path = "HI-Small_Trans.csv"
dataset = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"

# Prefer the dataset already present in the project cache.
local_cache = Path.cwd() / "kagglehub_cache"
local_cache.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLEHUB_CACHE"] = str(local_cache.resolve())

In [18]:
dataset_dir = local_cache / "datasets" / "ealtman2019" / "ibm-transactions-for-anti-money-laundering-aml" / "versions" / "8"
if not dataset_dir.exists():
    print("Local cache missing; downloading dataset bundle...")
    kagglehub.dataset_download(dataset)

available_files = sorted(path.name for path in dataset_dir.iterdir())
print("Dataset dir:", dataset_dir)
print("Available files:", available_files)

if file_path not in available_files:
    raise FileNotFoundError(
        f"{file_path} was not found. Pick one of: {available_files}"
    )

df = pd.read_csv(dataset_dir / file_path)
print(df.head())

Dataset dir: c:\Users\mamin\Desktop\GenAI-Genesis\kagglehub_cache\datasets\ealtman2019\ibm-transactions-for-anti-money-laundering-aml\versions\8
Available files: ['HI-Large_Patterns.txt', 'HI-Large_Trans.csv', 'HI-Large_accounts.csv', 'HI-Medium_Patterns.txt', 'HI-Medium_Trans.csv', 'HI-Medium_accounts.csv', 'HI-Small_Patterns.txt', 'HI-Small_Trans.csv', 'HI-Small_accounts.csv', 'LI-Large_Patterns.txt', 'LI-Large_Trans.csv', 'LI-Large_accounts.csv', 'LI-Medium_Patterns.txt', 'LI-Medium_Trans.csv', 'LI-Medium_accounts.csv', 'LI-Small_Patterns.txt', 'LI-Small_Trans.csv', 'LI-Small_accounts.csv']
          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:20         10  8000EBD30       10  8000EBD30   
1  2022/09/01 00:20       3208  8000F4580        1  8000F5340   
2  2022/09/01 00:00       3209  8000F4670     3209  8000F4670   
3  2022/09/01 00:02         12  8000F5030       12  8000F5030   
4  2022/09/01 00:06         10  8000F5200       10  8000F5200   

   Amoun

(5177, 11)

In [24]:
df.shape

(5078345, 11)

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

In [27]:
# --- Feature engineering ---
X = df[
    [
        "Timestamp",
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
        "Amount Received",
        "Receiving Currency",
        "Amount Paid",
        "Payment Currency",
        "Payment Format",
    ]
].copy()
y = df["Is Laundering"].astype(int)

# Parse timestamp into numeric parts
ts = pd.to_datetime(X["Timestamp"], errors="coerce")
X["Hour"] = ts.dt.hour.fillna(-1).astype("int16")
X["DayOfWeek"] = ts.dt.dayofweek.fillna(-1).astype("int16")
X["Day"] = ts.dt.day.fillna(-1).astype("int16")
X["Month"] = ts.dt.month.fillna(-1).astype("int16")
X = X.drop(columns=["Timestamp"])

# Encode categorical/text columns with integer codes
cat_cols = ["Account", "Account.1", "Receiving Currency", "Payment Currency", "Payment Format"]
for col in cat_cols:
    X[col] = pd.factorize(X[col], sort=True)[0].astype("int32")

# Optional downsampling for faster training on very large data
max_rows = 300_000
if len(X) > max_rows:
    frac = max_rows / len(X)
    sampled_idx = y.groupby(y, group_keys=False).apply(
        lambda s: s.sample(frac=frac, random_state=42)
    ).index
    X_model = X.loc[sampled_idx]
    y_model = y.loc[sampled_idx]
else:
    X_model, y_model = X, y

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42,
    stratify=y_model,
)

# Random Forest model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
rf.fit(X_train, y_train)

# Predictions and evaluation
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.6f}")
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, y_prob):.6f}")

Train shape: (240000, 13) | Test shape: (60000, 13)

Classification Report:
              precision    recall  f1-score   support

           0     0.9990    1.0000    0.9995     59939
           1     1.0000    0.0328    0.0635        61

    accuracy                         0.9990     60000
   macro avg     0.9995    0.5164    0.5315     60000
weighted avg     0.9990    0.9990    0.9986     60000

Confusion Matrix:
[[59939     0]
 [   59     2]]

ROC-AUC: 0.862057
PR-AUC (Average Precision): 0.223312


In [ ]:
Write me the code to train a random forest algorithm to flag money laundering. Here are the columns in df: "['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering']", Make sure to print evaluations in the end